# Swap calibration diagnostics

The neighbour-exchange channel and the off-channel (spurious) minimum depth are
numpy-only. The over-coordination trap search needs JAX (`stability_jax`).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from vbc_landscape import (swap_landscape, find_swap_endpoints, minimax_barrier,
                           naive_spurious_depth, sporadic_min_depth, spurious_depth)

# base potential
R_REP, R_ATT, E_R, E_A = 1.0, 1.5, 20.0, -2.0
BASE = dict(r_rep=R_REP, r_att=R_ATT, E_r=E_R, E_a=E_A)

# the three RCS-paper gates  (r_low, r_high, n_target);  n_target = 1 - n_delta
SETUPS = [("standard  VBC(1.5, 2.0, 0)",        (1.5,   2.0,   1.00)),
          ("looser  VBC(1.224, 1.561, -0.03)",  (1.224, 1.561, 1.03)),
          ("optimised  VBC(1.29, 1.50, 0.30)",  (1.29,  1.50,  0.70))]
VALS = [2, 3, 4, 5, 6]
r_grid = np.linspace(0.9, 2.2, 401)

## Neighbour-exchange channel

`U(r_A, r_B)` for the central particle, with the two swap minima (red) and the minimax
barrier joining them.

In [ ]:
def draw_channel(ax, valency, gate):
    rl, rh, nt = gate
    U = swap_landscape(valency, r_grid, rl, rh, nt, **BASE)
    ext = [r_grid[0], r_grid[-1], r_grid[0], r_grid[-1]]
    im = ax.imshow((U / abs(E_A)).T, origin="lower", extent=ext, aspect="equal",
                   vmin=-valency, vmax=0.0, cmap="viridis")
    title = f"f = {valency}"
    ep = find_swap_endpoints(U, r_grid)
    if ep is not None:
        B = minimax_barrier(U, ep[0], ep[1])
        ax.contour(r_grid, r_grid, U.T, levels=[B], colors="r", linewidths=1.0)
        for (i, j) in ep:
            ax.plot(r_grid[i], r_grid[j], "r*", ms=12)
        title += f"   B/|E_a| = {(B - max(U[ep[0]], U[ep[1]])) / abs(E_A):.3f}"
    ax.set_xlabel(r"$r_A$")
    ax.set_ylabel(r"$r_B$")
    ax.set_title(title)
    return im

gate = SETUPS[2][1]      # optimised
fig, axes = plt.subplots(1, 3, figsize=(12.5, 4.0))
for ax, v in zip(axes, [2, 4, 6]):
    im = draw_channel(ax, v, gate)
fig.colorbar(im, ax=list(axes), label=r"$U / |E_a|$", shrink=0.85)
plt.show()

## Off-channel (spurious) minimum depth

Depth of the trap felt by a particle approaching an already-saturated neighbour, three
estimates. `naive_spurious_depth` (bare pair, overstated), `sporadic_min_depth`
(corrected total-cluster energy), and `spurious_depth(key='random')` (corrected,
averaged over random neighbour placements).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.0), sharey=True)
for ax, (name, gate) in zip(axes, SETUPS):
    rl, rh, nt = gate
    naive = [naive_spurious_depth(f, rl, rh, nt, **BASE) for f in VALS]
    det   = [sporadic_min_depth(f, rl, rh, nt, **BASE) for f in VALS]
    rand  = [spurious_depth(f, rl, rh, nt, key="random", n_sample=200, seed=0, **BASE) for f in VALS]
    ax.plot(VALS, naive, "o-", label="naive (bare pair)")
    ax.plot(VALS, det, "s-", label="deterministic")
    ax.plot(VALS, rand, "^-", label="random")
    ax.set_xlabel("valency f")
    ax.set_title(name)
    ax.legend(fontsize=7)
axes[0].set_ylabel(r"spurious depth $/|E_a|$")
plt.tight_layout()
plt.show()

## Over-coordination trap search (needs JAX)

A saturated centre plus one extra particle, `m = valency + 1` neighbour radii.
`find_minima` relaxes to the stationary points of the total energy and keeps the
strictly convex interior ones. an interior minimum means the extra particle stays
trapped with all `valency + 1` bonds. The symmetric-energy curve below shows the well
for the looser gate and its absence for the others.

In [ ]:
try:
    import jax  # noqa: F401
    import jax.numpy as jnp
    from stability_jax import cluster_energy, find_minima, optimal_state_energy, curvature_scale
    HAVE_JAX = True
except ModuleNotFoundError:
    HAVE_JAX = False
    print("JAX not installed - skipping the trap search.  pip install 'jax[cpu]'  (see README).")

if HAVE_JAX:
    f = 6; m = f + 1
    rr = np.linspace(R_REP, R_ATT, 400)
    fig, ax = plt.subplots(figsize=(6.2, 4.0))
    for name, (rl, rh, nt) in SETUPS:
        ncut = f + (nt - 1.0)
        U = [float(cluster_energy(jnp.full(m, r), r_low=rl, r_high=rh, ncut=ncut, **BASE)) for r in rr]
        ax.plot(rr, np.array(U) / abs(E_A), label=name)
    ax.axhline(0.0, color="k", lw=0.8, ls="--")
    ax.set_xlabel(r"$r$   (all $m=f{+}1$ neighbours at radius $r$)")
    ax.set_ylabel(r"$U / |E_a|$")
    ax.set_title(f"over-coordinated symmetric energy (f={f}, m={m})")
    ax.legend(fontsize=7)
    plt.show()

In [ ]:
if HAVE_JAX:
    k0 = curvature_scale(E_A, R_REP, R_ATT)
    for name, (rl, rh, nt) in SETUPS:
        for f in VALS:
            mins = find_minima(f, rl, rh, nt, n_seed=100, **BASE)
            if mins:
                x, U, lam = mins[0]
                print(f"  {name} f={f}: TRAP  r~{x[0]:.3f}  U/|E_a|={U/abs(E_A):+.3f}  lam_min/k0={lam/k0:+.3f}")
            else:
                print(f"  {name} f={f}: none (extra particle rejected)")

## Depth of the looser-gate trap

Trap energy relative to the ideal `f`-bonded state (`-f|E_a|`, the standard gate's
optimum). negative = the extra particle is net bound, positive = a metastable pocket.

In [ ]:
if HAVE_JAX:
    std, looser = SETUPS[0][1], SETUPS[1][1]
    print(f"  {'f':>2}  {'U_trap(f+1)/|E_a|':>17}  {'ideal f/|E_a|':>13}  {'depth/|E_a|':>12}")
    depth = []
    for f in VALS:
        ideal = optimal_state_energy(f, *std, **BASE) / abs(E_A)
        mins = find_minima(f, *looser, n_seed=100, **BASE)
        if not mins:
            depth.append(np.nan)
            print(f"  {f:>2}  {'(no trap)':>17}  {ideal:>13.3f}")
            continue
        Ut = mins[0][1] / abs(E_A)
        depth.append(Ut - ideal)
        print(f"  {f:>2}  {Ut:>17.3f}  {ideal:>13.3f}  {Ut-ideal:>+12.3f}")
    fig, ax = plt.subplots(figsize=(5.5, 3.6))
    ax.plot(VALS, depth, "o-")
    ax.axhline(0.0, color="k", lw=0.8, ls="--")
    ax.set_xlabel("valency f")
    ax.set_ylabel(r"$(U_{\rm trap}(f{+}1) - U_{\rm ideal}(f)) / |E_a|$")
    ax.set_title("looser gate: over-coordination trap depth")
    plt.show()